In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from tiny_diffusion import datasets
from tiny_diffusion.model import Block, NoiseScheduler

### forward

In [ ]:
num_timesteps = 50
plot_step = 5

num_plots = math.ceil(num_timesteps / plot_step)
num_cols = 5
num_rows = math.ceil(num_plots / num_cols)

fig = plt.figure(figsize=(15, 6))

noise_scheduler = NoiseScheduler(num_timesteps=num_timesteps)
dataset = datasets.get_dataset("dino", n=1000)
x0 = dataset.tensors[0]

plt_cnt = 1
plt.subplot(num_rows, num_cols, plt_cnt)
plt.scatter(x0[:, 0], x0[:, 1], alpha=0.5, s=15)
plt.title("data")
plt.xlim(-3.5, 3.5)
plt.ylim(-4., 4.75)
plt.axis("off")

for t in range(len(noise_scheduler)):
    timesteps = torch.full((len(x0),), t, dtype=torch.long)
    noise = torch.randn_like(x0)
    sample = noise_scheduler.add_noise(x0, noise, timesteps)
    if (t + 1) % plot_step == 0 and (t + 1) != len(noise_scheduler):
        plt_cnt += 1
        plt.subplot(num_rows, num_cols, plt_cnt)
        plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15)
        plt.title(f"step: {t + 1}")
        plt.xlim(-3.5, 3.5)
        plt.ylim(-4., 4.75)
        plt.axis("off")

fig.tight_layout()
plt.savefig("static/forward.png", facecolor="white")
plt.show()

### reverse

In [ ]:
!uv run python -m tiny_diffusion.ddpm --experiment_name dino_base

In [ ]:
model = Block()

path = "data/output/dino_base/model.pth"
model.load_state_dict(torch.load(path))
model.eval()

In [ ]:
eval_batch_size = 1000
num_timesteps = 50
plot_step = 5
noise_scheduler = NoiseScheduler(num_timesteps=num_timesteps)
sample = torch.randn(eval_batch_size, 2)
timesteps = list(range(num_timesteps))[::-1]
samples = []
steps = []
for i, t in enumerate(tqdm(timesteps)):
    t_batch = torch.full((eval_batch_size,), t, dtype=torch.long)
    with torch.no_grad():
        residual = model(sample, t_batch)
    sample = noise_scheduler.step(residual, t_batch[0], sample)
    if (i + 1) % plot_step == 0:
        samples.append(sample.numpy())
        steps.append(i + 1)

In [ ]:
num_cols = 5
num_rows = math.ceil(len(samples) / num_cols)
fig = plt.figure(figsize=(15, 6))
for i, sample in enumerate(samples):
    plt.subplot(num_rows, num_cols, i + 1)
    plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15)
    plt.title(f"step: {steps[i]}")
    plt.xlim(-3.5, 3.5)
    plt.ylim(-4., 4.75)
    plt.axis("off")
fig.tight_layout()
plt.savefig("static/reverse.png", facecolor="white")
plt.show()

## ablations

In [ ]:
def plot_ablation(frames_dict, outname):
    num_rows = len(frames_dict)
    num_cols = 10

    plt.figure(figsize=(3.5*num_cols, 3*num_rows + 0.5))

    for row, (name, frames) in enumerate(frames_dict.items()):
        epoch_step = len(frames) // num_cols
        offset = row*(num_cols + 1)
        plt.subplot(num_rows, num_cols + 1, offset + 1)
        plt.scatter(0, 0, alpha=0)
        plt.text(0, 0, name, fontdict={"size": 30})
        plt.xlim(-0.25, 2)
        plt.axis("off")

        for i in range(num_cols):
            plt.subplot(num_rows, num_cols + 1, offset + i + 2)
            ix = i * epoch_step
            frame = frames[ix]
            plt.scatter(frame[:, 0], frame[:, 1], s=5, alpha=0.7)
            if row == 0:
                title = f"epoch {ix}" if i == 0 else f"{ix}"
                plt.title(title, fontdict={"size": 30}, pad=30)
            plt.xlim(-3.5, 3.5)
            plt.ylim(-4., 4.75)
            plt.axis("off")

    plt.tight_layout()
    plt.savefig(outname, facecolor="white")
    plt.show()

### datasets

In [ ]:
!uv run python -m tiny_diffusion.ddpm --dataset moons --experiment_name moons_base
!uv run python -m tiny_diffusion.ddpm --dataset dino --experiment_name dino_base
!uv run python -m tiny_diffusion.ddpm --dataset line --experiment_name line_base
!uv run python -m tiny_diffusion.ddpm --dataset circle --experiment_name circle_base

In [ ]:
frames_dict = {
    "moons": np.load("data/output/moons_base/frames.npy"),
    "dino": np.load("data/output/dino_base/frames.npy"),
    "line": np.load("data/output/line_base/frames.npy"),
    "circle": np.load("data/output/circle_base/frames.npy"),
}

plot_ablation(frames_dict, "static/datasets.png")

### learning rate

In [ ]:
!uv run python -m tiny_diffusion.ddpm --learning_rate 1e-2 --experiment_name dino_lr1e-2
!uv run python -m tiny_diffusion.ddpm --learning_rate 1e-3 --experiment_name dino_lr1e-3
!uv run python -m tiny_diffusion.ddpm --learning_rate 1e-4 --experiment_name dino_lr1e-4
!uv run python -m tiny_diffusion.ddpm --learning_rate 1e-5 --experiment_name dino_lr1e-5

In [ ]:
frames_dict = {
    "lr 1e-2": np.load("data/output/dino_lr1e-2/frames.npy"),
    "lr 1e-3": np.load("data/output/dino_lr1e-3/frames.npy"),
    "lr 1e-4": np.load("data/output/dino_lr1e-4/frames.npy"),
    "lr 1e-5": np.load("data/output/dino_lr1e-5/frames.npy"),
}

plot_ablation(frames_dict, "static/learning_rate.png")

### num_timesteps

In [ ]:
!uv run python -m tiny_diffusion.ddpm --num_timesteps 5 --experiment_name dino_timesteps5
!uv run python -m tiny_diffusion.ddpm --num_timesteps 10 --experiment_name dino_timesteps10
!uv run python -m tiny_diffusion.ddpm --num_timesteps 25 --experiment_name dino_timesteps25
!uv run python -m tiny_diffusion.ddpm --num_timesteps 50 --experiment_name dino_timesteps50
!uv run python -m tiny_diffusion.ddpm --num_timesteps 100 --experiment_name dino_timesteps100
!uv run python -m tiny_diffusion.ddpm --num_timesteps 250 --experiment_name dino_timesteps250

In [ ]:
frames_dict = {
    "        5": np.load("data/output/dino_timesteps5/frames.npy"),
    "       10": np.load("data/output/dino_timesteps10/frames.npy"),
    "       25": np.load("data/output/dino_timesteps25/frames.npy"),
    "       50": np.load("data/output/dino_timesteps50/frames.npy"),
    "      100": np.load("data/output/dino_timesteps100/frames.npy"),
    "      250": np.load("data/output/dino_timesteps250/frames.npy"),
}

plot_ablation(frames_dict, "static/num_timesteps.png")

### beta schedule

In [ ]:
!uv run python -m tiny_diffusion.ddpm --beta_schedule quadratic --experiment_name dino_quadratic_schedule

In [ ]:
frames_dict = {
    "linear": np.load("data/output/dino_base/frames.npy"),
    "quadratic": np.load("data/output/dino_quadratic_schedule/frames.npy"),
}

plot_ablation(frames_dict, "static/beta_schedule.png")

### hidden size

In [ ]:
!uv run python -m tiny_diffusion.ddpm --hidden_size 16 --experiment_name dino_hid_size_16
!uv run python -m tiny_diffusion.ddpm --hidden_size 32 --experiment_name dino_hid_size_32
!uv run python -m tiny_diffusion.ddpm --hidden_size 64 --experiment_name dino_hid_size_64
!uv run python -m tiny_diffusion.ddpm --hidden_size 256 --experiment_name dino_hid_size_256
!uv run python -m tiny_diffusion.ddpm --hidden_size 512 --experiment_name dino_hid_size_512

In [ ]:
frames_dict = {
    "       16": np.load("data/output/dino_hid_size_16/frames.npy"),
    "       32": np.load("data/output/dino_hid_size_32/frames.npy"),
    "       64": np.load("data/output/dino_hid_size_64/frames.npy"),
    "      128": np.load("data/output/dino_base/frames.npy"),
    "      256": np.load("data/output/dino_hid_size_256/frames.npy"),
    "      512": np.load("data/output/dino_hid_size_512/frames.npy"),
}

plot_ablation(frames_dict, "static/hidden_size.png")

### num layers

In [ ]:
!uv run python -m tiny_diffusion.ddpm --hidden_layers 1 --experiment_name dino_hid_layers_1
!uv run python -m tiny_diffusion.ddpm --hidden_layers 2 --experiment_name dino_hid_layers_2
!uv run python -m tiny_diffusion.ddpm --hidden_layers 4 --experiment_name dino_hid_layers_4
!uv run python -m tiny_diffusion.ddpm --hidden_layers 5 --experiment_name dino_hid_layers_5

In [ ]:
frames_dict = {
    "        1": np.load("data/output/dino_hid_layers_1/frames.npy"),
    "        2": np.load("data/output/dino_hid_layers_2/frames.npy"),
    "        3": np.load("data/output/dino_base/frames.npy"),
    "        4": np.load("data/output/dino_hid_layers_4/frames.npy"),
    "        5": np.load("data/output/dino_hid_layers_5/frames.npy"),

}

plot_ablation(frames_dict, "static/num_hidden_layers.png")

### positional embedding (timestep)

In [ ]:
!uv run python -m tiny_diffusion.ddpm --time_embedding learnable --experiment_name dino_time_emb_learnable
!uv run python -m tiny_diffusion.ddpm --time_embedding linear --experiment_name dino_time_emb_linear
!uv run python -m tiny_diffusion.ddpm --time_embedding zero --experiment_name dino_time_emb_zeros

In [ ]:
frames_dict = {
    "learnable": np.load("data/output/dino_time_emb_learnable/frames.npy"),
    "sinusoidal": np.load("data/output/dino_base/frames.npy"),
    "linear": np.load("data/output/dino_time_emb_linear/frames.npy"),
    "zero": np.load("data/output/dino_time_emb_zeros/frames.npy"),
}

plot_ablation(frames_dict, "static/time_embedding.png")

### positional embedding (inputs)

In [ ]:
!uv run python -m tiny_diffusion.ddpm --input_embedding learnable --experiment_name dino_input_emb_learnable
!uv run python -m tiny_diffusion.ddpm --input_embedding linear --experiment_name dino_input_emb_linear
!uv run python -m tiny_diffusion.ddpm --input_embedding identity --experiment_name dino_input_emb_identity

In [ ]:
frames_dict = {
    "learnable": np.load("data/output/dino_input_emb_learnable/frames.npy"),
    "sinusoidal": np.load("data/output/dino_base/frames.npy"),
    "linear": np.load("data/output/dino_input_emb_linear/frames.npy"),
    "identity": np.load("data/output/dino_input_emb_identity/frames.npy"),
}

plot_ablation(frames_dict, "static/input_embedding.png")

### forward and reverse process animation

In [ ]:
from celluloid import Camera

In [ ]:
num_timesteps = 250
noise_scheduler = NoiseScheduler(num_timesteps=num_timesteps)

model = Block()
path = "data/output/dino_timesteps250/model.pth"
model.load_state_dict(torch.load(path))
model.eval()

dataset = datasets.get_dataset("dino", n=1000)
x0 = dataset.tensors[0]

In [ ]:
forward_samples = []
forward_samples.append(x0)
for t in range(len(noise_scheduler)):
    timesteps = torch.full((len(x0),), t, dtype=torch.long)
    noise = torch.randn_like(x0)
    sample = noise_scheduler.add_noise(x0, noise, timesteps)
    forward_samples.append(sample)

In [ ]:
eval_batch_size = len(dataset)
sample = torch.randn(eval_batch_size, 2)
timesteps = list(range(num_timesteps))[::-1]
reverse_samples = []
reverse_samples.append(sample.numpy())
for _i, t in enumerate(tqdm(timesteps)):
    t_batch = torch.full((eval_batch_size,), t, dtype=torch.long)
    with torch.no_grad():
        residual = model(sample, t_batch)
    sample = noise_scheduler.step(residual, t_batch[0], sample)
    reverse_samples.append(sample.numpy())

In [ ]:
xmin, xmax = -3.5, 3.5
ymin, ymax = -4., 4.75

fig, ax = plt.subplots()
camera = Camera(fig)

# forward
for i, sample in enumerate(forward_samples):
    plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15, color="blue")
    ax.text(0.0, 0.95, f"step {i: 4} / {num_timesteps}", transform=ax.transAxes)
    ax.text(0.0, 1.01, "Forward process", transform=ax.transAxes, size=15)
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)
    plt.axis("off")
    camera.snap()

# reverse
for i, sample in enumerate(reverse_samples):
    plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15, color="blue")
    ax.text(0.0, 0.95, f"step {i: 4} / {num_timesteps}", transform=ax.transAxes)
    ax.text(0.0, 1.01, "Reverse process", transform=ax.transAxes, size=15)
    plt.xlim(xmin, xmax)
    plt.ylim(ymin, ymax)
    plt.axis("off")
    camera.snap()

animation = camera.animate(blit=True, interval=35)
animation.save("static/animation.mp4")